<a href="https://colab.research.google.com/github/Kminhas21/ds2002-fa26/blob/main/2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item, meaning one item on one order (so an order with 2 items shows up as 2 rows).

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
assert len(lines) == 4
assert lines['qty'].sum() == 7
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors = pd.DataFrame(vendors_json)
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)
unmatched = joined[joined['_merge'] == 'left_only']
print('unmatched vendor:', unmatched['vendor_id'].unique().tolist())
print('units attached:', unmatched['qty'].sum())
joined

unmatched vendor: ['V-07']
units attached: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# Keep the unmatched vendor as its own labeled row (instead of dropping it) so the totals still add up to 7.
joined['vendor_label'] = joined['name'].fillna(joined['vendor_id'] + ' (unmatched)')
qty_by_vendor = joined.groupby('vendor_label')['qty'].sum()
print(qty_by_vendor)
assert qty_by_vendor.sum() == 7

vendor_label
Cav Merch           1
Hoos Burgers        3
V-07 (unmatched)    3
Name: qty, dtype: int64


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# dropna=False keeps the zone-less rows visible instead of silently losing them
zone_totals = joined.groupby('zone', dropna=False)['qty'].sum()
print(zone_totals)
print('zone A total:', zone_totals['A'], '| sum of known zones:', joined.dropna(subset=['zone'])['qty'].sum())
# Zone A sold the most (4 units). Known zones total 4, not 7: the other 3 units belong to unmatched vendor V-07, which has no zone (NaN).

zone
A      4
NaN    3
Name: qty, dtype: int64
zone A total: 4 | sum of known zones: 4


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# json_normalize(record_path='lines') raises a KeyError on order 5, so unpack by hand with .get()
# An order with no lines becomes a single row with item/qty missing (not dropped).
rows = []
for o in ragged_json:
    for ln in (o.get('lines') or [{}]):
        rows.append({'order': o['order'], 'vendor_id': o['vendor_id'],
                     'item': ln.get('item'), 'qty': ln.get('qty')})
ragged = pd.DataFrame(rows)
ragged['qty'] = ragged['qty'].astype('Int64')   # nullable int: missing stays <NA>, not 0
print(ragged)
print('total qty (NA skipped):', ragged['qty'].sum(), '| rows with missing qty:', ragged['qty'].isna().sum())

   order vendor_id   item   qty
0      4      V-01   Soda     2
1      5      V-10   None  <NA>
2      6      V-01  Fries  <NA>
total qty (NA skipped): 2 | rows with missing qty: 2


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** One row per line item is useful because the line item is the thing that actually has a quantity, so you can group by item, vendor, or zone with a plain `groupby` and sum. With one row per order, the items are buried in a list, so questions like "how many units did each vendor sell" would need extra unpacking first.

**b)** A missing quantity means we don't know the amount, while zero means we know nothing was sold. If you conflate them with `fillna(0)`, totals and averages look precise when they are really undercounts, and you lose the ability to flag which orders need to be fixed upstream.